# `styxx.audit_confound` — is your classifier's score tracking the concept, or a confound?

Every deployed text scorer — a toxicity filter, a sentiment model, an LLM guardrail — can separate its
training data while secretly keying on a **confound** (length, politeness, identity terms…) that happened
to correlate with the label. When the confound and the concept come apart in production, it fails silently.

`audit_confound` finds out on a corpus where the concept and the suspected confound are **orthogonal**, and
returns a CI-backed verdict — `ROBUST` / `THRESHOLD-BIASED` (+ a validated `report.guard()`) /
`CONFOUND-DEPENDENT` / `INCONCLUSIVE`.

**New in 7.23.0 — the substrate gate.** A verdict is only as trustworthy as the corpus it's measured on. If
that corpus is LLM-*generated*, the generator can entangle the confound with the concept and **manufacture**
the very bias you're testing for. So any alarming verdict on a non-ground-truth corpus now carries a
`SYNTHETIC-ARTIFACT RISK` flag, and `styxx.validate_against_ground_truth` re-runs the audit on real
human-labeled data. We learned this by falsifying our *own* published result — this notebook shows both.


In [ ]:
%pip install -q 'styxx[hf]'


## 1 · Catch a planted confound (instant, no downloads)
A scorer whose value = `3·label − 2·confound`. It *can* tell the classes apart, but its score rides the
confound. The auditor flags it `THRESHOLD-BIASED` and hands back a length-fair `guard()`.


In [ ]:
import numpy as np
from styxx import audit_confound

rng = np.random.default_rng(0); n = 200
y = np.tile([0, 1], n // 2)                 # the concept label
confound = rng.standard_normal(n)           # a cue, orthogonal to the label
score = 3.0 * y - 2.0 * confound + rng.normal(0, 0.3, n)   # a scorer that rides the confound
rows = [{'text': f'item {i}', 'label': int(y[i]), 'confound': float(confound[i])} for i in range(n)]

# this is a controlled demo with a KNOWN, planted cue -> tell the gate it's ground truth so it
# doesn't (correctly) warn that a generator might have manufactured the effect.
rep = audit_confound(rows, scores=list(score), instrument='my_classifier', confound='my_cue',
                     corpus_provenance='ground_truth')
print(rep.verdict)
print('\nscore at reference cue:', round(rep.guard(1.0, rep.guard_ref), 3),
      '| same score, extreme cue, corrected:', round(rep.guard(1.0, rep.guard_ref + 3), 3))


## 2 · Audit a real model — and watch the tool flag its own synthetic test set
`distilbert-base-uncased-finetuned-sst-2-english` is the **default HuggingFace sentiment model**.
`audit_hf_model` loads it, scores a bundled boundary corpus, and audits — in one call.

The bundled corpus is **LLM-generated**, so the verdict comes back `THRESHOLD-BIASED` **with a**
`SYNTHETIC-ARTIFACT RISK` **flag**: a label-trained bag-of-words itself rides length on this corpus, which
is the fingerprint of a generator-injected confound — *not* proof the model is biased.

This is exactly what happened to our own 9-model Confound Report Card: it flagged classifiers as
length-biased on generated text, then **none of it replicated on real human-labeled reviews**
([the finding](https://github.com/fathom-lab/styxx/blob/main/papers/grounded-honesty-axis/FINDING_groundtruth_substrate_artifact_2026_06_27.md)).


In [ ]:
from styxx import audit_hf_model

rep = audit_hf_model('distilbert-base-uncased-finetuned-sst-2-english', construct='sentiment')
print(rep.verdict)
print('\nsynthetic_artifact_warning:', rep.synthetic_artifact_warning)
print('lexical entanglement  corr / perm-p:', rep.lexical_confound_corr, '/', rep.lexical_confound_p)
print('within-stratum AUC:', rep.within_stratum_auc, '| confound coef:', rep.confound_score_coef,
      rep.confound_score_coef_ci95)


## 3 · Validate the verdict on ground truth
A synthetic-corpus flag is a **hypothesis**. To confirm or kill it, re-run the *identical* audit on REAL
human-labeled data, length-matched by coarsened-exact-matching (orthogonality by *selection*, not
generation). `validate_against_ground_truth` does the matching, the re-audit, and the reconciliation:
`SYNTHETIC-ARTIFACT (refuted)` / `CONFIRMED` / `REAL-ONLY` / `CLEAR` / `INCONCLUSIVE`.


In [ ]:
from styxx import validate_against_ground_truth, cem_length_match

# Bring REAL human-labeled rows: {'text': str, 'label': 0/1, 'confound': <number, e.g. log word count>}.
# e.g. via `datasets`: Yelp/Amazon 2-star (neg) vs 4-star (pos) for boundary difficulty, or Civil Comments
# for toxicity. Then score each real text with the SAME model and reconcile:
#
#   real_rows = [{'text': r['text'], 'label': r['label'],
#                 'confound': float(np.log1p(len(r['text'].split())))} for r in my_real_data]
#   real_report, reconciliation = validate_against_ground_truth(
#       rep, real_rows, score_fn=lambda t: p_positive_from_distilbert(t))
#   print(reconciliation)
#
# (On our reviews, every synthetic sentiment flag reconciled to SYNTHETIC-ARTIFACT (refuted).)
print('Bring real human-labeled rows to confirm or refute the synthetic verdict.')


## 4 · Audit YOUR classifier
**A — you have labeled text + a cue:** make `rows` of `{text, label(0/1), confound(number)}`, pass your
model as `score_fn`, and set `corpus_provenance='ground_truth'` if the labels are real (human) labels.

**B — generate an orthogonal grid with an LLM:** `build_confound_grid` crosses two stance prompts with
confound levels. Convenient, but the verdict will be flagged `SYNTHETIC-ARTIFACT RISK` — validate it on real
data before trusting it.


In [ ]:
from styxx import audit_confound, build_confound_grid

# --- A: your own (ideally real-labeled) rows ---
# def my_model(text): return float(...)   # your classifier's score for `text`
# rows = [{'text': t, 'label': lbl, 'confound': len(t.split())} for t, lbl in my_data]
# report = audit_confound(rows, score_fn=my_model, confound='length', corpus_provenance='ground_truth')
# print(report.verdict)
# if report.verdict.startswith('THRESHOLD'):
#     fair = report.guard(raw_score, confound_value)   # confound-fair score

# --- B: generate the grid with your LLM (then VALIDATE on ground truth) ---
# rows = build_confound_grid(
#     items=['topic 1', 'topic 2', ...],
#     pos_prompt='You write clearly POSITIVE reviews.',
#     neg_prompt='You write clearly NEGATIVE reviews.',
#     confound_rules={'short': 'One sentence.', 'long': 'Five sentences.'},
#     generate_fn=my_llm)
# report = audit_confound(rows, score_fn=my_model, confound='log_words')  # -> SYNTHETIC-ARTIFACT RISK
print('Fill in A or B above with your classifier.')


---
The lesson, in one line: **if you audit a model on generated data, validate the verdict on ground truth —
or you may be measuring your generator, not your model.**

More: the finding `papers/grounded-honesty-axis/FINDING_groundtruth_substrate_artifact_2026_06_27.md` ·
repo [github.com/fathom-lab/styxx](https://github.com/fathom-lab/styxx).
